# Portfolio Drift Detection Agent System
## Microsoft Foundry - Private Banking Pre-Review Stage

This system detects portfolio drift and geopolitical market impacts, alerting Relationship Managers (RMs) to actionable rebalancing opportunities.

### 1. Install Required Libraries

In [1]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity yfinance requests pandas numpy urllib3

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\zvijayfa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
%pip install skfolio pyportfolioopt vectorbt backtrader scipy scikit-learn pandas-ta yfinance requests pandas numpy urllib3 -q

### 2. Setup and Environment Configuration

In [3]:
import os
import json
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

openai_client = project_client.get_openai_client()

print("✅ Environment configured successfully")

✅ Environment configured successfully


### 3. Client Portfolio Data with Target Allocations (Feb 2026 Baseline)

In [4]:
# Portfolio data as of Feb 2026
client_portfolios = {
    "Client_1": {
        "name": "Tech-Heavy Growth Client",
        "rm_name": "Relationship Manager A",
        "risk_profile": "High",
        "portfolio_value": 5000000,  # ₹50 Lakhs
        "target_allocation": {
            "IT_stocks": 0.45,      # 45% - IT heavy
            "banking_stocks": 0.20,  # 20% - Banking
            "energy_stocks": 0.15,   # 15% - Energy (affected by Iran-USA war)
            "mutual_funds": 0.15,    # 15% - Diversification
            "cash": 0.05             # 5% - Reserve
        },
        "current_allocation": {
            "IT_stocks": 0.45,
            "banking_stocks": 0.20,
            "energy_stocks": 0.15,
            "mutual_funds": 0.15,
            "cash": 0.05
        },
        "holdings": {
            "IT": ["TCS.NS", "WIPRO.NS", "INFY.NS"],
            "Banking": ["HDFCBANK.NS"],
            "Energy": ["RELIANCE.NS", "NTPC.NS"],
            "MutualFunds": ["119551", "119022"]  # HDFC Nifty 50, SBI Bluechip
        },
        "geopolitical_impact_sectors": ["Energy", "Banking"]
    },
    "Client_2": {
        "name": "Balanced Growth Client",
        "rm_name": "Relationship Manager B",
        "risk_profile": "Moderate",
        "portfolio_value": 7500000,  # ₹75 Lakhs
        "target_allocation": {
            "IT_stocks": 0.25,
            "banking_stocks": 0.25,
            "energy_stocks": 0.20,
            "mutual_funds": 0.20,
            "cash": 0.10
        },
        "current_allocation": {
            "IT_stocks": 0.25,
            "banking_stocks": 0.25,
            "energy_stocks": 0.20,
            "mutual_funds": 0.20,
            "cash": 0.10
        },
        "holdings": {
            "IT": ["INFY.NS", "TCS.NS"],
            "Banking": ["HDFCBANK.NS", "ICICIBANK.NS"],
            "Energy": ["RELIANCE.NS", "COALINDIA.NS"],
            "MutualFunds": ["119551", "120485"]
        },
        "geopolitical_impact_sectors": ["Energy"]
    },
    "Client_3": {
        "name": "Conservative Income Client",
        "rm_name": "Relationship Manager C",
        "risk_profile": "Low",
        "portfolio_value": 3000000,  # ₹30 Lakhs
        "target_allocation": {
            "IT_stocks": 0.10,
            "banking_stocks": 0.30,
            "energy_stocks": 0.15,
            "mutual_funds": 0.30,
            "cash": 0.15
        },
        "current_allocation": {
            "IT_stocks": 0.10,
            "banking_stocks": 0.30,
            "energy_stocks": 0.15,
            "mutual_funds": 0.30,
            "cash": 0.15
        },
        "holdings": {
            "IT": ["INFY.NS"],
            "Banking": ["HDFCBANK.NS", "ICICIBANK.NS"],
            "Energy": ["NTPC.NS", "POWERGRID.NS"],
            "MutualFunds": ["119547", "119551"]  # HDFC Dividend Yield
        },
        "geopolitical_impact_sectors": ["Energy", "Banking"]
    },
    "Client_4": {
        "name": "Emerging Market Growth Client",
        "rm_name": "Relationship Manager D",
        "risk_profile": "High",
        "portfolio_value": 4000000,  # ₹40 Lakhs
        "target_allocation": {
            "IT_stocks": 0.30,
            "banking_stocks": 0.15,
            "energy_stocks": 0.25,
            "mutual_funds": 0.20,
            "cash": 0.10
        },
        "current_allocation": {
            "IT_stocks": 0.30,
            "banking_stocks": 0.15,
            "energy_stocks": 0.25,
            "mutual_funds": 0.20,
            "cash": 0.10
        },
        "holdings": {
            "IT": ["WIPRO.NS", "TCS.NS"],
            "Banking": ["ICICIBANK.NS"],
            "Energy": ["RELIANCE.NS", "ADANIGREEN.NS", "NTPC.NS"],
            "MutualFunds": ["119598", "119618"]
        },
        "geopolitical_impact_sectors": ["Energy", "Banking"]
    },
    "Client_5": {
        "name": "Sector-Focused Specialist Client",
        "rm_name": "Relationship Manager E",
        "risk_profile": "Moderate-High",
        "portfolio_value": 6000000,  # ₹60 Lakhs
        "target_allocation": {
            "IT_stocks": 0.40,
            "banking_stocks": 0.15,
            "energy_stocks": 0.20,
            "mutual_funds": 0.15,
            "cash": 0.10
        },
        "current_allocation": {
            "IT_stocks": 0.40,
            "banking_stocks": 0.15,
            "energy_stocks": 0.20,
            "mutual_funds": 0.15,
            "cash": 0.10
        },
        "holdings": {
            "IT": ["TCS.NS", "INFY.NS", "WIPRO.NS"],
            "Banking": ["HDFCBANK.NS"],
            "Energy": ["RELIANCE.NS", "COALINDIA.NS"],
            "MutualFunds": ["119510", "120303"]
        },
        "geopolitical_impact_sectors": ["Energy", "Banking"]
    }
}

print("✅ Portfolio data loaded for 5 clients")
for client_id, data in client_portfolios.items():
    print(f"   {data['name']} - RM: {data['rm_name']}")

✅ Portfolio data loaded for 5 clients
   Tech-Heavy Growth Client - RM: Relationship Manager A
   Balanced Growth Client - RM: Relationship Manager B
   Conservative Income Client - RM: Relationship Manager C
   Emerging Market Growth Client - RM: Relationship Manager D
   Sector-Focused Specialist Client - RM: Relationship Manager E


### 4. Portfolio Drift Detection Functions

### 4A. Advanced Finance Analytics: PyPortfolioOpt & Skfolio Integration

In [ ]:
from pypfopt import EfficientFrontier, mean_historical_return, sample_cov, plot_efficient_frontier
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

def fetch_portfolio_historical_data(holdings_dict, end_date=None, period="6mo"):
    """Fetch historical OHLCV data for all holdings up to Feb 2026"""
    if end_date is None:
        end_date = pd.Timestamp("2026-02-28")
    
    all_returns = {}
    prices_data = {}
    
    for sector, tickers in holdings_dict.items():
        if sector == "MutualFunds":
            continue
        
        sector_returns = []
        for ticker in tickers:
            try:
                tick_obj = yf.Ticker(ticker)
                hist = tick_obj.history(period=period, end=end_date)
                
                if not hist.empty and len(hist) > 1:
                    daily_returns = hist['Close'].pct_change().dropna()
                    all_returns[ticker] = daily_returns
                    prices_data[ticker] = hist['Close']
                    
            except Exception as e:
                print(f"Error fetching {ticker}: {str(e)[:50]}")
    
    return all_returns, prices_data

def calculate_portfolio_statistics(returns_dict):
    """Calculate advanced portfolio statistics using scipy"""
    if not returns_dict:
        return None
    
    # Combine all returns into a DataFrame
    returns_df = pd.DataFrame(returns_dict)
    
    stats_dict = {
        'mean_return': returns_df.mean() * 252,  # Annualized
        'std_dev': returns_df.std() * np.sqrt(252),  # Annualized
        'correlation_matrix': returns_df.corr(),
        'covariance_matrix': returns_df.cov() * 252,
        'skewness': returns_df.skew(),
        'kurtosis': returns_df.kurtosis(),
        'sharpe_ratio': (returns_df.mean() * 252) / (returns_df.std() * np.sqrt(252))
    }
    
    # Value at Risk (VaR) calculation
    for ticker in returns_df.columns:
        stats_dict[f'var_95_{ticker}'] = returns_df[ticker].quantile(0.05)
    
    return stats_dict, returns_df

def optimize_portfolio_efficient_frontier(returns_df, target_return=None):
    """Optimize portfolio using Efficient Frontier (Modern Portfolio Theory)"""
    try:
        mu = mean_historical_return(returns_df)
        S = sample_cov(returns_df)
        
        ef = EfficientFrontier(mu, S)
        
        if target_return:
            ef.efficient_return(target_return)
        else:
            ef.min_volatility()  # Minimize volatility
        
        weights = ef.clean_weights()
        perf = ef.portfolio_performance(verbose=False)
        
        return {
            'weights': weights,
            'expected_return': perf[0],
            'volatility': perf[1],
            'sharpe_ratio': perf[2],
            'ticker_weights': {ticker: weights[ticker] for ticker in weights if weights[ticker] > 0.01}
        }
    except Exception as e:
        print(f"Optimization error: {str(e)[:100]}")
        return None

def calculate_portfolio_risk_metrics(prices_df, weights, lookback_days=30):
    """Calculate advanced risk metrics: VaR, CVaR, Drawdown"""
    portfolio_returns = (prices_df.pct_change() * np.array(list(weights.values()))).sum(axis=1)
    
    # Value at Risk (95% confidence)
    var_95 = portfolio_returns.quantile(0.05)
    
    # Conditional Value at Risk (Expected Shortfall)
    cvar_95 = portfolio_returns[portfolio_returns <= var_95].mean()
    
    # Maximum Drawdown
    cumulative = (1 + portfolio_returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    
    return {
        'var_95': var_95,
        'cvar_95': cvar_95,
        'max_drawdown': max_drawdown,
        'sortino_ratio': (portfolio_returns.mean() * 252) / (portfolio_returns[portfolio_returns < 0].std() * np.sqrt(252))
    }

print("✅ Advanced Portfolio Optimization functions loaded")
print("   - PyPortfolioOpt: Efficient Frontier optimization")
print("   - SciPy: Risk metrics and statistical analysis")
print("   - Correlation & Covariance analysis")

### 4B. VectorBT Backtesting & Scenario Simulation

In [ ]:
import vectorbt as vbt

def simulate_crisis_scenarios(prices_df, baseline_date="2026-02-28"):
    """
    Simulate crisis scenarios for Apr 2026:
    1. IT Sector decline: -8% to -12%
    2. Energy sector decline: -5% to -7% (geopolitical)
    3. Banking volatility: -3% to -5%
    """
    
    crisis_date = pd.Timestamp("2026-04-30")
    crisis_scenarios = {}
    
    # Get IT stocks (TCS, WIPRO, INFY)
    it_stocks = [col for col in prices_df.columns if any(x in col for x in ['TCS', 'WIPRO', 'INFY'])]
    energy_stocks = [col for col in prices_df.columns if any(x in col for x in ['RELIANCE', 'NTPC', 'COAL'])]
    banking_stocks = [col for col in prices_df.columns if any(x in col for x in ['HDFC', 'ICICI'])]
    
    scenarios_config = {
        'baseline': {},  # No changes
        'it_crisis_8pct': {stock: -0.08 for stock in it_stocks},
        'it_crisis_12pct': {stock: -0.12 for stock in it_stocks},
        'geopolitical_crisis': {stock: -0.06 for stock in energy_stocks},
        'combined_crisis': {**{stock: -0.10 for stock in it_stocks}, 
                           **{stock: -0.06 for stock in energy_stocks}},
        'severe_crisis': {**{stock: -0.12 for stock in it_stocks},
                         **{stock: -0.07 for stock in energy_stocks},
                         **{stock: -0.05 for stock in banking_stocks}}
    }
    
    for scenario_name, shock_dict in scenarios_config.items():
        scenario_prices = prices_df.copy()
        
        # Apply shocks to baseline prices
        for ticker, shock in shock_dict.items():
            if ticker in scenario_prices.columns:
                scenario_prices[ticker] = scenario_prices[ticker] * (1 + shock)
        
        returns = scenario_prices.pct_change().dropna()
        
        crisis_scenarios[scenario_name] = {
            'prices': scenario_prices,
            'returns': returns,
            'cumulative_return': (1 + returns).cumprod() - 1,
            'volatility': returns.std() * np.sqrt(252),
            'max_drawdown': ((1 + returns).cumprod() / (1 + returns).cumprod().cummax() - 1).min()
        }
    
    return crisis_scenarios

def backtest_portfolio_performance(prices_df, weights, scenarios=None):
    """Backtest portfolio performance across different scenarios"""
    
    results = {}
    
    for ticker, weight in weights.items():
        if ticker in prices_df.columns:
            results[ticker] = weight
    
    # Calculate portfolio returns
    portfolio_returns = (prices_df[list(weights.keys())].pct_change() * np.array(list(weights.values()))).sum(axis=1)
    
    backtest_metrics = {
        'total_return': (1 + portfolio_returns).prod() - 1,
        'annualized_return': portfolio_returns.mean() * 252,
        'volatility': portfolio_returns.std() * np.sqrt(252),
        'sharpe_ratio': (portfolio_returns.mean() * 252) / (portfolio_returns.std() * np.sqrt(252)) if portfolio_returns.std() > 0 else 0,
        'max_drawdown': ((1 + portfolio_returns).cumprod() / (1 + portfolio_returns).cumprod().cummax() - 1).min(),
        'win_rate': len(portfolio_returns[portfolio_returns > 0]) / len(portfolio_returns),
        'calmar_ratio': (portfolio_returns.mean() * 252) / abs(((1 + portfolio_returns).cumprod() / (1 + portfolio_returns).cumprod().cummax() - 1).min())
    }
    
    return backtest_metrics

def analyze_portfolio_correlation_stress(correlation_matrix, threshold=0.7):
    """
    Analyze portfolio correlation stress:
    - High correlation = correlation risk
    - Market crisis tends to increase correlations
    """
    
    high_corr_pairs = []
    
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            corr_value = correlation_matrix.iloc[i, j]
            
            if abs(corr_value) > threshold:
                high_corr_pairs.append({
                    'pair': f"{correlation_matrix.columns[i]} - {correlation_matrix.columns[j]}",
                    'correlation': corr_value,
                    'diversification_risk': 'HIGH' if abs(corr_value) > 0.8 else 'MEDIUM'
                })
    
    return {
        'high_correlation_pairs': high_corr_pairs,
        'avg_portfolio_correlation': correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].mean(),
        'diversification_benefit': 1 - correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].mean()
    }

print("✅ VectorBT Backtesting functions loaded")
print("   - Crisis scenario simulation (IT -8-12%, Energy -5-7%)")
print("   - Portfolio performance backtesting")
print("   - Correlation stress analysis")

### 4C. Feb 2026 Historical Data Analysis & Portfolio Baseline

In [ ]:
# Feb 2026 Baseline Analysis (6 months of historical data)
feb_2026_end = pd.Timestamp("2026-02-28")

client_optimization_data = {}

for client_id, portfolio in client_portfolios.items():
    print(f"\n{'='*80}")
    print(f"Analyzing {client_id}: {portfolio['name']}")
    print(f"{'='*80}")
    
    # Fetch historical data for Feb 2026 baseline
    returns_dict, prices_data = fetch_portfolio_historical_data(
        portfolio['holdings'], 
        end_date=feb_2026_end, 
        period="6mo"
    )
    
    if not returns_dict:
        print(f"⚠️ Insufficient data for {client_id}")
        continue
    
    # Calculate portfolio statistics
    stats_result = calculate_portfolio_statistics(returns_dict)
    if stats_result:
        portfolio_stats, returns_df = stats_result
        
        # Efficient Frontier Optimization
        opt_result = optimize_portfolio_efficient_frontier(returns_df)
        
        if opt_result:
            # Calculate risk metrics
            combined_prices = pd.concat(prices_data.values(), axis=1, keys=prices_data.keys())
            risk_metrics = calculate_portfolio_risk_metrics(combined_prices, opt_result['weights'])
            
            client_optimization_data[client_id] = {
                'stats': portfolio_stats,
                'returns_df': returns_df,
                'optimal_weights': opt_result['ticker_weights'],
                'expected_return': opt_result['expected_return'],
                'volatility': opt_result['volatility'],
                'sharpe_ratio': opt_result['sharpe_ratio'],
                'risk_metrics': risk_metrics,
                'current_allocation': portfolio['target_allocation']
            }
            
            print(f"✅ Feb 2026 Baseline Performance:")
            print(f"   Expected Return (Annualized): {opt_result['expected_return']*100:.2f}%")
            print(f"   Volatility (Annualized): {opt_result['volatility']*100:.2f}%")
            print(f"   Sharpe Ratio: {opt_result['sharpe_ratio']:.3f}")
            print(f"   Value at Risk (95%): {risk_metrics['var_95']*100:.2f}%")
            print(f"   Max Drawdown: {risk_metrics['max_drawdown']*100:.2f}%")
            print(f"   Sortino Ratio: {risk_metrics['sortino_ratio']:.3f}")

print(f"\n{'='*80}")
print(f"✅ Feb 2026 Baseline Analysis Complete for {len(client_optimization_data)} clients")
print(f"{'='*80}")

### 4D. Apr 2026 Crisis Scenario Simulation & Stress Testing

In [ ]:
# Apr 2026 Crisis Scenarios - Stress Test All Portfolios
crisis_simulations = {}

for client_id in client_optimization_data.keys():
    print(f"\n{'='*100}")
    print(f"CRISIS SCENARIO ANALYSIS: {client_id}")
    print(f"{'='*100}")
    
    portfolio = client_portfolios[client_id]
    returns_df = client_optimization_data[client_id]['returns_df']
    
    # Create combined price data from returns
    prices_df = (1 + returns_df).cumprod()
    
    # Run crisis simulations
    scenarios = simulate_crisis_scenarios(prices_df, baseline_date="2026-02-28")
    
    scenario_impact = {}
    
    for scenario_name, scenario_data in scenarios.items():
        current_weights = portfolio['target_allocation'].copy()
        
        # Convert sector allocations to stock-level weights
        stock_weights = {}
        for sector, pct in current_weights.items():
            if sector in portfolio['holdings']:
                stocks = [s for s in portfolio['holdings'][sector] if s in returns_df.columns]
                for stock in stocks:
                    stock_weights[stock] = pct / len(stocks)
        
        # Backtest under this scenario
        backtest_result = backtest_portfolio_performance(
            scenario_data['prices'], 
            stock_weights
        )
        
        scenario_impact[scenario_name] = {
            **backtest_result,
            'volatility_impact': (backtest_result['volatility'] - 
                                 client_optimization_data[client_id]['volatility']) / 
                                 client_optimization_data[client_id]['volatility'] * 100,
            'return_impact': (backtest_result['annualized_return'] - 
                             client_optimization_data[client_id]['expected_return']) / 
                             abs(client_optimization_data[client_id]['expected_return'] + 0.001) * 100
        }
    
    crisis_simulations[client_id] = scenario_impact
    
    # Print scenario analysis
    for scenario, impact in scenario_impact.items():
        print(f"\n📊 {scenario.upper()}")
        print(f"   Return Impact: {impact['return_impact']:+.2f}%")
        print(f"   Volatility Impact: {impact['volatility_impact']:+.2f}%")
        print(f"   Sharpe Ratio: {impact['sharpe_ratio']:.3f}")
        print(f"   Max Drawdown: {impact['max_drawdown']*100:.2f}%")

print(f"\n{'='*100}")
print(f"✅ Crisis Scenario Analysis Complete")
print(f"   Scenarios: 6 (Baseline, IT -8%, IT -12%, Geopolitical, Combined, Severe)")
print(f"{'='*100}")

In [5]:
def get_sector_performance(sector_stocks, sector_name):
    """Calculate sector performance metrics"""
    try:
        sector_prices = []
        for stock in sector_stocks:
            try:
                ticker = yf.Ticker(stock)
                hist = ticker.history(period="3mo")
                if not hist.empty:
                    current_price = hist['Close'].iloc[-1]
                    price_30days_ago = hist['Close'].iloc[0] if len(hist) > 0 else current_price
                    change_pct = ((current_price - price_30days_ago) / price_30days_ago * 100) if price_30days_ago != 0 else 0
                    sector_prices.append({
                        'stock': stock,
                        'current_price': current_price,
                        'change_percent': change_pct
                    })
            except:
                pass
        
        if sector_prices:
            avg_change = np.mean([p['change_percent'] for p in sector_prices])
            return {
                'sector': sector_name,
                'stocks': sector_prices,
                'average_change_percent': round(avg_change, 2),
                'performance': 'UNDERPERFORMING' if avg_change < -5 else 'NORMAL' if avg_change < 5 else 'OUTPERFORMING'
            }
    except Exception as e:
        print(f"Error fetching {sector_name} performance: {e}")
    
    return None

def calculate_portfolio_drift(client_id, sector_performances):
    """Calculate portfolio drift based on sector performance"""
    portfolio = client_portfolios.get(client_id)
    if not portfolio:
        return None
    
    drift_analysis = {
        'client_id': client_id,
        'client_name': portfolio['name'],
        'portfolio_value': portfolio['portfolio_value'],
        'target_allocation': portfolio['target_allocation'],
        'drift_detected': [],
        'total_drift_risk': 0,
        'alert_level': 'LOW',
        'recommended_actions': []
    }
    
    for perf in sector_performances:
        sector_name = list(portfolio['holdings'].keys())[list(perf['sector'].lower().startswith(s.lower()) for s in ['it', 'banking', 'energy', 'mutual']).index(True)] if any(perf['sector'].lower().startswith(s.lower()) for s in ['it', 'banking', 'energy']) else None
        
        if perf['performance'] == 'UNDERPERFORMING':
            allocated_pct = portfolio['target_allocation'].get(sector_name + '_stocks', 0) if sector_name else 0
            if allocated_pct > 0.15:  # Significant allocation
                drift_detected = {
                    'sector': perf['sector'],
                    'current_performance': perf['average_change_percent'],
                    'allocated_percentage': allocated_pct * 100,
                    'drift_amount_rupees': portfolio['portfolio_value'] * allocated_pct * (abs(perf['average_change_percent']) / 100),
                    'impact': 'HIGH' if perf['average_change_percent'] < -10 else 'MEDIUM' if perf['average_change_percent'] < -5 else 'LOW'
                }
                drift_analysis['drift_detected'].append(drift_detected)
    
    # Calculate total drift and alert level
    if drift_analysis['drift_detected']:
        total_drift_value = sum([d['drift_amount_rupees'] for d in drift_analysis['drift_detected']])
        drift_analysis['total_drift_risk'] = total_drift_value
        
        if total_drift_value > portfolio['portfolio_value'] * 0.10:  # > 10% portfolio impact
            drift_analysis['alert_level'] = 'CRITICAL'
        elif total_drift_value > portfolio['portfolio_value'] * 0.05:  # > 5% portfolio impact
            drift_analysis['alert_level'] = 'HIGH'
        else:
            drift_analysis['alert_level'] = 'MEDIUM'
        
        # Generate recommendations
        for drift in drift_analysis['drift_detected']:
            action = f"Rebalance {drift['sector']}: Reduce from {drift['allocated_percentage']:.1f}% by ₹{drift['drift_amount_rupees']:.0f} due to {abs(drift['current_performance']):.1f}% underperformance"
            drift_analysis['recommended_actions'].append(action)
    else:
        drift_analysis['alert_level'] = 'LOW'
    
    return drift_analysis

print("✅ Portfolio Drift Detection functions loaded")

✅ Portfolio Drift Detection functions loaded


### 5. Create Drift Detection Agent Instructions

In [6]:
def create_drift_agent_instructions(client_id):
    """Create specialized instructions for Portfolio Drift Detection Agent"""
    portfolio = client_portfolios.get(client_id)
    if not portfolio:
        return None
    
    holdings_summary = ", ".join([f"{sector}: {', '.join(stocks)}" for sector, stocks in portfolio['holdings'].items()])
    affected_sectors = ", ".join(portfolio['geopolitical_impact_sectors'])
    
    instructions = f"""
You are a Portfolio Drift Detection Agent for {portfolio['name']}.
Your primary mission is to identify portfolio drift and alert the Relationship Manager to actionable rebalancing opportunities.

CLIENT INFORMATION:
==================
Client ID: {client_id}
Relationship Manager: {portfolio['rm_name']}
Risk Profile: {portfolio['risk_profile']}
Portfolio Value: ₹{portfolio['portfolio_value']:,}

TARGET ALLOCATION (Feb 2026 Baseline):
========================================
- IT Stocks: {portfolio['target_allocation']['IT_stocks']*100:.0f}%
- Banking Stocks: {portfolio['target_allocation']['banking_stocks']*100:.0f}%
- Energy Stocks: {portfolio['target_allocation']['energy_stocks']*100:.0f}%
- Mutual Funds: {portfolio['target_allocation']['mutual_funds']*100:.0f}%
- Cash Reserve: {portfolio['target_allocation']['cash']*100:.0f}%

CURRENT HOLDINGS:
=================
{holdings_summary}

GEOPOLITICAL RISK FACTORS (April 2026):
=========================================
Recent Market Events:
1. INDIAN IT SECTOR DECLINE: Claude AI release triggered tech stock sell-off (TCS, WIPRO, INFY down 8-12%)
2. IRAN-USA GEOPOLITICAL TENSION: Energy sector affected ({affected_sectors})
3. BANKING SECTOR VOLATILITY: Interest rate uncertainty impacting financial stocks

PRIORITY SECTORS TO MONITOR: {affected_sectors}

YOUR MISSION:
=============
1. CONTINUOUS MONITORING: Track real-time portfolio drift from target allocation
2. DRIFT CALCULATION: 
   - Monitor each sector's performance vs 3-month baseline
   - Flag if any sector underperforms by >5%
   - Calculate portfolio impact in rupees

3. ALERT GENERATION:
   - CRITICAL: Portfolio impact >10% OR sector underperforming >10%
   - HIGH: Portfolio impact 5-10% OR sector underperforming 5-10%
   - MEDIUM: Portfolio impact 2-5% OR sector underperforming 2-5%
   - LOW: Minimal impact, continue monitoring

4. ACTIONABLE RECOMMENDATIONS:
   - Specify WHICH sectors to reduce
   - Specify HOW MUCH to move (in rupees and percentage)
   - Specify WHERE to move funds (safer alternatives, fixed income, cash)
   - Estimate tax implications
   - Provide urgency level

5. CONTEXT PROVIDED:
   - Market indices for benchmarking (Nifty 50, Nifty IT, Nifty Energy)
   - Individual stock performance within sectors
   - Historical volatility patterns
   - Life event triggers (if provided via Dynamics 365)

RESPONSE FORMAT FOR RM:
=======================
When providing recommendations:

[ALERT STATUS]: [CRITICAL/HIGH/MEDIUM/LOW]
[TIMESTAMP]: Current market snapshot date
[AFFECTED SECTORS]: Which sectors are drifting
[PORTFOLIO IMPACT]: ₹X impact (~Y% of portfolio)
[RECOMMENDED ACTION]: Specific rebalancing instruction
[URGENCY]: Take action within X days
[TAX CONSIDERATION]: Any capital gains implications

COMMUNICATION STYLE:
====================
- Be direct and actionable (RM must understand immediately)
- Use precise numbers (rupees, percentages)
- Provide specific stock/sector names
- Relate to geopolitical events impacting markets
- Always provide alternative recommendations
- Highlight both risks and opportunities

IMPORTANT:
===========
- This is PRE-REVIEW preparation (before RM calls client)
- Your job is to arm the RM with data-driven insights
- Flag subtle drifts early (5% threshold)
- Consider this client's specific risk tolerance
- Link portfolio changes to current world events
- Always provide at least 2 rebalancing options
"""
    
    return instructions.strip()

print("✅ Drift Agent Instructions template created")

✅ Drift Agent Instructions template created


### 5A. Enhanced Drift Detection Agent Instructions with Optimization

In [ ]:
def create_enhanced_drift_agent_instructions(client_id, opt_data=None, crisis_sim=None):
    """
    Create enhanced agent instructions incorporating:
    - Portfolio optimization recommendations
    - Crisis scenario analysis
    - Risk metrics and stress test results
    - Modern Portfolio Theory insights
    """
    
    portfolio = client_portfolios.get(client_id)
    if not portfolio:
        return None
    
    holdings_summary = ", ".join([f"{sector}: {', '.join(stocks)}" for sector, stocks in portfolio['holdings'].items()])
    affected_sectors = ", ".join(portfolio['geopolitical_impact_sectors'])
    
    # Build optimization context
    opt_context = ""
    if opt_data and client_id in opt_data:
        opt = opt_data[client_id]
        opt_context = f"""
PORTFOLIO OPTIMIZATION ANALYSIS (Feb 2026 Baseline):
====================================================
Modern Portfolio Theory Recommendations:

Current Risk Metrics:
- Expected Annual Return: {opt['expected_return']*100:.2f}%
- Volatility (Annual): {opt['volatility']*100:.2f}%
- Sharpe Ratio: {opt['sharpe_ratio']:.3f}
- Value at Risk (95%): {opt['risk_metrics']['var_95']*100:.2f}% daily loss
- Maximum Drawdown: {opt['risk_metrics']['max_drawdown']*100:.2f}%
- Sortino Ratio: {opt['risk_metrics']['sortino_ratio']:.3f}

Optimal Allocation (from Efficient Frontier):
{chr(10).join([f"  - {ticker}: {pct*100:.1f}%" for ticker, pct in list(opt['optimal_weights'].items())[:5]])}

Key Insight: Current vs. Optimal
- Consider rebalancing to optimize risk-adjusted returns
- Focus on Sharpe Ratio maximization
- Reduce concentration risk through diversification
"""
    
    # Build crisis analysis context
    crisis_context = ""
    if crisis_sim and client_id in crisis_sim:
        crisis = crisis_sim[client_id]
        worst_case = min(crisis.items(), key=lambda x: x[1]['sharpe_ratio'])
        
        crisis_context = f"""
CRISIS SCENARIO STRESS TEST (Apr 2026):
========================================
Scenario Analysis Results:

Worst Case: {worst_case[0].upper()}
- Impact on Return: {worst_case[1]['return_impact']:+.2f}%
- Impact on Volatility: {worst_case[1]['volatility_impact']:+.2f}%
- Projected Sharpe: {worst_case[1]['sharpe_ratio']:.3f}
- Max Drawdown: {worst_case[1]['max_drawdown']*100:.2f}%

Portfolio Resilience Score: {100 * (1 + min(c['return_impact'] for c in crisis.values()) / 100):.1f}%

Recommendation: If crisis occurs, prepare contingency rebalancing strategy
"""
    
    instructions = f"""
You are an Advanced Portfolio Drift Detection & Optimization Agent for {portfolio['name']}.
Your mission goes beyond drift detection to include active portfolio optimization and scenario planning.

CLIENT INFORMATION:
==================
Client ID: {client_id}
Relationship Manager: {portfolio['rm_name']}
Risk Profile: {portfolio['risk_profile']}
Portfolio Value: ₹{portfolio['portfolio_value']:,}

TARGET ALLOCATION (Feb 2026 Baseline):
========================================
- IT Stocks: {portfolio['target_allocation']['IT_stocks']*100:.0f}%
- Banking Stocks: {portfolio['target_allocation']['banking_stocks']*100:.0f}%
- Energy Stocks: {portfolio['target_allocation']['energy_stocks']*100:.0f}%
- Mutual Funds: {portfolio['target_allocation']['mutual_funds']*100:.0f}%
- Cash Reserve: {portfolio['target_allocation']['cash']*100:.0f}%

CURRENT HOLDINGS:
=================
{holdings_summary}

{opt_context}

{crisis_context}

GEOPOLITICAL RISK FACTORS (April 2026):
=========================================
Recent Market Events:
1. INDIAN IT SECTOR DECLINE: Claude AI release triggered tech stock sell-off (TCS, WIPRO, INFY down 8-12%)
2. IRAN-USA GEOPOLITICAL TENSION: Energy sector affected ({affected_sectors})
3. BANKING SECTOR VOLATILITY: Interest rate uncertainty impacting financial stocks

PRIORITY SECTORS TO MONITOR: {affected_sectors}

YOUR ADVANCED MISSION:
======================
1. CONTINUOUS MONITORING: Track real-time portfolio drift from optimal allocation
2. MULTI-DIMENSIONAL ANALYSIS:
   - Drift calculation vs target allocation
   - Risk metrics monitoring (VaR, Sharpe, Drawdown)
   - Correlation stress analysis
   - Scenario impact projections

3. OPTIMIZATION-BASED RECOMMENDATIONS:
   - Compare current vs optimal weights (from Efficient Frontier)
   - Calculate rebalancing impact on Sharpe ratio
   - Estimate tax consequences
   - Provide recovery timeline

4. CRISIS SCENARIO RESPONSE:
   - Alert level: CRITICAL if portfolio deviation >10% from efficient frontier
   - Suggest dynamic rebalancing based on stress test results
   - Recommend hedging strategies if correlations spike

5. RM PRE-REVIEW DASHBOARD:
   - Drift status vs two baselines (target and optimal)
   - Projected portfolio performance under crisis
   - Specific action items with expected outcomes
   - Risk-adjusted return optimization path

RESPONSE FORMAT FOR RM:
=======================
[ALERT STATUS]: [CRITICAL/HIGH/MEDIUM/LOW]
[DRIFT TYPE]: Target-based OR Optimization-based
[CURRENT PERFORMANCE]: Annualized return, volatility, Sharpe
[SCENARIO IMPACT]: Expected impact if crisis occurs
[RECOMMENDED ACTION]: Specific rebalancing instruction with expected outcome
[OPTIMIZATION BENEFIT]: Projected Sharpe improvement if executed
[URGENCY]: Immediate/Within 1 week/Monitor

COMMUNICATION STYLE:
====================
- Be direct and data-driven (RM needs precise numbers)
- Link all recommendations to risk metrics (Sharpe, VaR, Drawdown)
- Provide probability-weighted scenario outcomes
- Always give 2+ actionable alternatives
- Quantify expected improvement from each action

CRITICAL:
==========
- This is PRE-REVIEW preparation (before RM calls client)
- Your job is to provide optimal recommendations, not just detect drift
- Use Modern Portfolio Theory insights
- Consider behavioral psychology in implementation (avoid tax-inefficient trades)
- Flag early (3% drift from optimal) to allow proactive management
- Always relate to current geopolitical/market events
"""
    
    return instructions.strip()

# Generate enhanced instructions for all clients
enhanced_instructions = {}
for client_id in client_portfolios.keys():
    enhanced_instructions[client_id] = create_enhanced_drift_agent_instructions(
        client_id, 
        client_optimization_data, 
        crisis_simulations
    )

print("✅ Enhanced Drift Agent Instructions with Optimization Created")
print(f"   Features: Modern Portfolio Theory + Crisis Scenarios + Risk Metrics")
print(f"   Clients: {len(enhanced_instructions)}")

### 6. Create Foundry Drift Detection Agents

### 5B. Multi-Agent Workflow Architecture (Agent Orchestration)

In [ ]:
"""
MULTI-AGENT WORKFLOW ARCHITECTURE FOR PORTFOLIO ANALYSIS
=========================================================

Workflow Stages:
1. DATA-AGENT: Collects market data, computes statistics
2. OPTIMIZATION-AGENT: Runs Modern Portfolio Theory optimization
3. RISK-AGENT: Performs stress testing and crisis scenario analysis
4. DRIFT-DETECTOR-AGENT: Identifies deviations and alerts
5. RECOMMENDATION-AGENT: Synthesizes insights and provides actions
6. AGGREGATOR-AGENT: Collects all recommendations for RM dashboard

Agent Interaction Pattern (Sequential Workflow):
INPUT → [Data-Agent] → [Optimization-Agent] → [Risk-Agent] → 
         [Drift-Detector] → [Recommendation-Agent] → [Aggregator] → OUTPUT

For parallel analysis:
[Data-Agent] ─┬─→ [Optimization-Agent]
              ├─→ [Risk-Agent]
              └─→ [Drift-Detector] ─→ [Recommendation-Agent] → [Aggregator]
"""

class MultiAgentPortfolioAnalyzer:
    """Orchestrates multiple specialized agents for comprehensive portfolio analysis"""
    
    def __init__(self, project_client, openai_client, model_deployment_name):
        self.project_client = project_client
        self.openai_client = openai_client
        self.model_deployment_name = model_deployment_name
        self.agents = {}
        self.conversations = {}
    
    def create_specialized_agents(self):
        """Create specialized agents for different analysis tasks"""
        
        agents_config = {
            'data-agent': {
                'name': 'portfolio-data-analyst',
                'instructions': """You are a Portfolio Data Analysis Agent.
Your role: Fetch and analyze market data, compute statistical metrics (returns, volatility, correlation).
Provide: Clean data summary, statistical overview, data quality assessment.
Output format: {"data_quality": "high/medium", "statistics": {...}, "anomalies": [...]}"""
            },
            
            'optimization-agent': {
                'name': 'portfolio-optimizer',
                'instructions': """You are a Modern Portfolio Theory Optimization Agent.
Your role: Execute Efficient Frontier optimization, compute optimal weights, analyze risk-return tradeoff.
Use: PyPortfolioOpt for Efficient Frontier, calculate Sharpe ratio, VaR, CVaR.
Output format: {"optimal_weights": {...}, "sharpe_ratio": X, "expected_return": Y%, "volatility": Z%}"""
            },
            
            'risk-agent': {
                'name': 'portfolio-risk-analyst',
                'instructions': """You are a Portfolio Risk Analysis Agent.
Your role: Perform stress testing, run crisis scenarios, analyze correlation stress, compute drawdown.
Scenarios: IT sector -8-12%, Energy -5-7%, Banking -3-5%, Combined crisis.
Output format: {"scenarios": {...}, "risk_metrics": {...}, "resilience_score": X%}"""
            },
            
            'drift-detector': {
                'name': 'portfolio-drift-detector',
                'instructions': """You are a Portfolio Drift Detection Agent.
Your role: Compare current allocation vs target and optimal, identify deviations, calculate impact.
Alert triggers: Drift >5% from target, >3% from optimal, correlation spike >0.8.
Output format: {"drift_status": "HIGH/MEDIUM/LOW", "deviations": {...}, "alert_level": "CRITICAL/HIGH/MEDIUM/LOW"}"""
            },
            
            'recommendation-agent': {
                'name': 'portfolio-recommendation-engine',
                'instructions': """You are a Portfolio Recommendation Synthesis Agent.
Your role: Integrate insights from all agents, provide actionable recommendations, quantify expected outcomes.
Consider: Tax efficiency, market timing, behavioral factors, RM communication style.
Output format: {"actions": [...], "expected_sharpe_improvement": X%, "urgency": "IMMEDIATE/WEEK/MONITOR"}"""
            }
        }
        
        for agent_type, config in agents_config.items():
            try:
                agent = self.project_client.agents.create_version(
                    agent_name=config['name'],
                    definition=PromptAgentDefinition(
                        model=self.model_deployment_name,
                        instructions=config['instructions']
                    )
                )
                self.agents[agent_type] = {
                    'agent': agent,
                    'agent_name': config['name'],
                    'agent_id': agent.id
                }
                print(f"✅ Created: {config['name']}")
            except Exception as e:
                print(f"⚠️ Error creating {config['name']}: {str(e)[:100]}")
    
    def orchestrate_workflow(self, client_id, portfolio_data):
        """Execute sequential workflow for comprehensive analysis"""
        
        workflow_results = {
            'client_id': client_id,
            'stages': {}
        }
        
        # Stage 1: Data Collection & Analysis
        if 'data-agent' in self.agents:
            try:
                conv_id = self.openai_client.conversations.create().id
                query = f"Analyze portfolio data for {client_id}: {portfolio_data}"
                
                response = self.openai_client.responses.create(
                    conversation=conv_id,
                    extra_body={
                        'agent': {
                            'name': self.agents['data-agent']['agent_name'],
                            'type': 'agent_reference'
                        }
                    },
                    input=query
                )
                workflow_results['stages']['data_analysis'] = response.output_text
            except Exception as e:
                workflow_results['stages']['data_analysis'] = f"Error: {str(e)[:100]}"
        
        return workflow_results

# Initialize Multi-Agent Analyzer
print("\n" + "="*100)
print("INITIALIZING MULTI-AGENT WORKFLOW ORCHESTRATION")
print("="*100)

try:
    analyzer = MultiAgentPortfolioAnalyzer(
        project_client, 
        openai_client, 
        model_deployment_name
    )
    
    print("\n🤖 Creating Specialized Agents...")
    analyzer.create_specialized_agents()
    
    print(f"\n✅ Multi-Agent System Ready with {len(analyzer.agents)} specialized agents")
    print("   Agents: Data-Analyst, Optimizer, Risk-Analyst, Drift-Detector, Recommendation-Engine")
except Exception as e:
    print(f"⚠️ Multi-agent initialization error: {str(e)[:150]}")

In [8]:
# Create Drift Detection Agents for each client
drift_agents = {}

for client_id, portfolio_data in client_portfolios.items():
    agent_name = f"drift-detector-{client_id.lower().replace('_', '-')}"
    instructions = create_drift_agent_instructions(client_id)
    
    agent = project_client.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions=instructions
        )
    )
    
    drift_agents[client_id] = {
        "agent": agent,
        "agent_name": agent_name,
        "portfolio": portfolio_data
    }
    
    print(f"✅ Drift Agent Created: {agent_name}")
    print(f"   Agent ID: {agent.id}")
    print(f"   RM: {portfolio_data['rm_name']}")
    print(f"   Risk Profile: {portfolio_data['risk_profile']}")
    print()

✅ Drift Agent Created: drift-detector-client-1
   Agent ID: drift-detector-client-1:1
   RM: Relationship Manager A
   Risk Profile: High

✅ Drift Agent Created: drift-detector-client-2
   Agent ID: drift-detector-client-2:1
   RM: Relationship Manager B
   Risk Profile: Moderate

✅ Drift Agent Created: drift-detector-client-3
   Agent ID: drift-detector-client-3:1
   RM: Relationship Manager C
   Risk Profile: Low

✅ Drift Agent Created: drift-detector-client-4
   Agent ID: drift-detector-client-4:1
   RM: Relationship Manager D
   Risk Profile: High

✅ Drift Agent Created: drift-detector-client-5
   Agent ID: drift-detector-client-5:1
   RM: Relationship Manager E
   Risk Profile: Moderate-High



### 7. Create Conversations for Each Agent

In [9]:
drift_conversations = {}

for client_id in drift_agents.keys():
    conversation = openai_client.conversations.create()
    drift_conversations[client_id] = conversation.id

print("✅ Conversations created for all Drift Detection Agents")

✅ Conversations created for all Drift Detection Agents


### 8. Query Drift Agents - Client 1 (Heavy IT Exposure - MOST IMPACTED)

In [10]:
# Client 1 has heavy IT exposure - most affected by Claude AI market shock
client_id = "Client_1"

query = """Current situation (April 2026):
- Anthropic released Claude co-work and Claude code with latest models
- This triggered a significant sell-off in Indian IT stocks (TCS, WIPRO, INFY down 8-12%)
- Additionally, Iran-USA geopolitical tensions have affected energy sector

Analyze my portfolio for drift. What should I do?"""

response = openai_client.responses.create(
    conversation=drift_conversations[client_id],
    extra_body={
        "agent": {
            "name": drift_agents[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=query
)

print("="*80)
print(f"PORTFOLIO DRIFT ALERT - {drift_agents[client_id]['portfolio']['name']}")
print(f"RM: {drift_agents[client_id]['portfolio']['rm_name']}")
print("="*80)
print(f"\nQuery Context: IT Sector Crisis + Geopolitical Risk")
print(f"\nAgent Analysis:\n{response.output_text}")
print("\n" + "="*80)

PORTFOLIO DRIFT ALERT - Tech-Heavy Growth Client
RM: Relationship Manager A

Query Context: IT Sector Crisis + Geopolitical Risk

Agent Analysis:
**[ALERT STATUS]: CRITICAL**  
**[TIMESTAMP]: April 2026**  
**[AFFECTED SECTORS]: IT stocks, Energy sector**  
**[PORTFOLIO IMPACT]: ₹710,000 impact (14.2% of portfolio)**  

---

### Detailed Drift Analysis:
1. **IT Sector (Target Allocation: 45%)**  
   - Recent 8-12% sell-off in TCS, WIPRO, and INFY has caused IT holdings to shrink significantly.  
   - Assuming equal allocation within IT stocks and highest decline (~12%), IT sector weight has dropped to ~39% (from 45%).  
   - Portfolio Value Impact: ~₹300,000 (~6% of ₹5,000,000 portfolio).

2. **Energy Sector (Target Allocation: 15%)**  
   - Geopolitical tension has introduced significant uncertainty, reducing investor confidence in Reliance and NTPC.  
   - Sector underperforming due to broader Iran-USA tension. Estimated decline ~8%, reducing allocation to ~13.8%.  
   - Portfolio Va

### 9. Query Drift Agents - Client 4 (Energy-Heavy - Secondary Impact)

In [11]:
client_id = "Client_4"

query = """Portfolio Drift Alert for April 2026:

Market Events:
1. Indian IT sector experiencing 8-12% decline due to Anthropic's Claude AI releases
2. Energy sector under pressure from Iran-USA geopolitical tensions
3. Banking sector showing volatility due to RBI rate uncertainty

My portfolio has significant energy exposure. Should I rebalance?"""

response = openai_client.responses.create(
    conversation=drift_conversations[client_id],
    extra_body={
        "agent": {
            "name": drift_agents[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=query
)

print("="*80)
print(f"PORTFOLIO DRIFT ALERT - {drift_agents[client_id]['portfolio']['name']}")
print(f"RM: {drift_agents[client_id]['portfolio']['rm_name']}")
print("="*80)
print(f"\nQuery Context: Geopolitical Risk + Energy Sector Pressure")
print(f"\nAgent Analysis:\n{response.output_text}")
print("\n" + "="*80)

PORTFOLIO DRIFT ALERT - Emerging Market Growth Client
RM: Relationship Manager D

Query Context: Geopolitical Risk + Energy Sector Pressure

Agent Analysis:
[ALERT STATUS]: **HIGH**
[TIMESTAMP]: April 2026, Market Snapshot

[AFFECTED SECTORS]:
1. **IT Stocks**: Down 8-12% due to Claude AI-driven tech sell-off.
2. **Energy Sector**: Geopolitical tensions impacting prices and volatility.
3. **Banking Sector**: Volatility from RBI interest rate uncertainty.

[PORTFOLIO IMPACT]:
- **IT Sector**: Decline reduces portfolio IT exposure from 30% target to ~27% (~₹1,080,000 → ₹1,080,000 × 0.92 = ₹993,600), causing a ₹86,400 loss (~2.16% of portfolio).
- **Energy Stocks**: Estimated exposure reduction from 25% target to ~22% (₹1,000,000 → ₹1,000,000 × 0.88 = ₹880,000), causing a ₹120,000 loss (~3% of portfolio).
- **Banking Stocks**: Slight drift projected (~15% target likely reduced to ~14%), but smaller impact (~₹60,000 loss/~1.5% overall).

Estimated total drift impact: **~₹266,400 loss (~6.6

### 10. Multi-Turn Conversation - Follow-Up Analysis

In [12]:
# Follow-up query in same conversation
client_id = "Client_1"

followup_query = """Based on your drift analysis, what are the specific rebalancing options?
Option 1: How much should I reduce IT stocks?
Option 2: What alternative sectors should I move to?
Option 3: Should I increase cash reserves given the uncertainty?"""

response = openai_client.responses.create(
    conversation=drift_conversations[client_id],
    extra_body={
        "agent": {
            "name": drift_agents[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=followup_query
)

print("="*80)
print(f"FOLLOW-UP ANALYSIS - {drift_agents[client_id]['portfolio']['name']}")
print("="*80)
print(f"\nFollow-up Query: Specific Rebalancing Options")
print(f"\nAgent Recommendations:\n{response.output_text}")
print("\n" + "="*80)

FOLLOW-UP ANALYSIS - Tech-Heavy Growth Client

Follow-up Query: Specific Rebalancing Options

Agent Recommendations:
### **Rebalancing Options Based on Portfolio Drift Analysis**

**Portfolio Context:**  
- IT stocks have underperformed **8-12%** due to competitive pressures from the latest AI models.  
- Energy stocks have declined **~8%**, affected by geopolitical tensions.  
- Significant portfolio drift of **~14.2%** (Critical Alert).  

Below are **three specific rebalancing options** addressing your concerns:

---

### **Option 1: How much to reduce IT stocks?**
#### Recommendation:
- Reduce overall IT stock exposure by **₹450,000 (9%)**.  
   - Rationale: IT stocks have already declined **8-12%**, and global AI competition is expected to sustain pressure in the near term. Gradually reducing allocation ensures proactive risk management.
   - Post-reduction, IT will move closer to **40% allocation** from the current **~39%**.  

#### Suggested Reductions Within IT:
1. **TCS.NS:** 

### 11. Generate RM Dashboard Summary

### 11A. Advanced Comprehensive Dashboard - Feb 2026 to Apr 2026 Analysis

In [ ]:
print("\n" + "="*120)
print("ADVANCED PORTFOLIO ANALYSIS DASHBOARD")
print("Feb 2026 (Baseline) → Apr 2026 (Crisis Scenario)")
print("Multi-Finance Package Integration: PyPortfolioOpt | VectorBT | SciPy | Skfolio")
print("="*120)

# Create comprehensive analysis for each client
comprehensive_analysis = []

for client_id in sorted(client_portfolios.keys()):
    portfolio = client_portfolios[client_id]
    
    if client_id not in client_optimization_data:
        continue
    
    opt_data = client_optimization_data[client_id]
    crisis_data = crisis_simulations.get(client_id, {})
    
    # Get worst-case scenario
    worst_scenario = min(crisis_data.items(), key=lambda x: x[1]['sharpe_ratio']) if crisis_data else ('N/A', {})
    
    analysis = {
        'Client': portfolio['name'],
        'RM': portfolio['rm_name'],
        'AUM': f"₹{portfolio['portfolio_value']/1000000:.1f}Cr",
        'Feb_2026_Return': f"{opt_data['expected_return']*100:.1f}%",
        'Feb_2026_Volatility': f"{opt_data['volatility']*100:.1f}%",
        'Feb_2026_Sharpe': f"{opt_data['sharpe_ratio']:.2f}",
        'VaR_95': f"{opt_data['risk_metrics']['var_95']*100:.2f}%",
        'Max_Drawdown': f"{opt_data['risk_metrics']['max_drawdown']*100:.1f}%",
        'Worst_Crisis': worst_scenario[0] if worst_scenario[0] != 'N/A' else 'N/A',
        'Crisis_Sharpe': f"{worst_scenario[1].get('sharpe_ratio', 0):.2f}" if worst_scenario[0] != 'N/A' else 'N/A',
        'Sharpe_Decline': f"{worst_scenario[1].get('return_impact', 0):+.1f}%" if worst_scenario[0] != 'N/A' else 'N/A'
    }
    
    comprehensive_analysis.append(analysis)

df_comprehensive = pd.DataFrame(comprehensive_analysis)
print("\n" + "📊 PORTFOLIO PERFORMANCE MATRIX (Feb 2026 vs Apr 2026 Crisis)")
print(df_comprehensive.to_string(index=False))

print("\n" + "="*120)
print("DETAILED CLIENT ANALYSIS")
print("="*120)

for client_id in sorted(client_portfolios.keys()):
    if client_id not in client_optimization_data:
        continue
    
    portfolio = client_portfolios[client_id]
    opt_data = client_optimization_data[client_id]
    crisis_data = crisis_simulations.get(client_id, {})
    
    print(f"\n🔷 {client_id}: {portfolio['name']}")
    print(f"   RM: {portfolio['rm_name']} | Risk: {portfolio['risk_profile']} | AUM: ₹{portfolio['portfolio_value']/1000000:.1f}Cr")
    
    print(f"\n   📈 FEB 2026 BASELINE METRICS:")
    print(f"      Expected Return (Annual): {opt_data['expected_return']*100:>6.2f}%")
    print(f"      Volatility (Annual):      {opt_data['volatility']*100:>6.2f}%")
    print(f"      Sharpe Ratio:             {opt_data['sharpe_ratio']:>6.3f}")
    print(f"      Value at Risk (95%):      {opt_data['risk_metrics']['var_95']*100:>6.2f}% daily max loss")
    print(f"      Max Drawdown:             {opt_data['risk_metrics']['max_drawdown']*100:>6.2f}%")
    print(f"      Sortino Ratio:            {opt_data['risk_metrics']['sortino_ratio']:>6.3f}")
    
    print(f"\n   🎯 OPTIMIZED ALLOCATION (Efficient Frontier):")
    for ticker, weight in sorted(opt_data['optimal_weights'].items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"      {ticker:>15}: {weight*100:>5.1f}%")
    
    if crisis_data:
        print(f"\n   ⚠️  CRISIS SCENARIO IMPACT (Apr 2026):")
        
        scenarios_sorted = sorted(crisis_data.items(), key=lambda x: x[1]['sharpe_ratio'])
        
        for scenario_name, scenario_metrics in scenarios_sorted[:3]:  # Show 3 worst scenarios
            impact_color = "🔴" if scenario_metrics['sharpe_ratio'] < 0 else "🟡" if scenario_metrics['sharpe_ratio'] < opt_data['sharpe_ratio']/2 else "🟢"
            print(f"      {impact_color} {scenario_name:>20}: Sharpe {scenario_metrics['sharpe_ratio']:>6.2f} | Return {scenario_metrics['return_impact']:+>6.1f}% | Max DD {scenario_metrics['max_drawdown']*100:>5.1f}%")
    
    print(f"\n   💡 RECOMMENDATION PRIORITY:")
    it_exposure = portfolio['target_allocation']['IT_stocks']
    energy_exposure = portfolio['target_allocation']['energy_stocks']
    
    if it_exposure > 0.30 or energy_exposure > 0.25:
        print(f"      🔴 CRITICAL: High sector concentration - Immediate rebalancing recommended")
    elif it_exposure > 0.20 or energy_exposure > 0.15:
        print(f"      🟠 HIGH: Moderate concentration - Plan rebalancing within 1 week")
    else:
        print(f"      🟡 MEDIUM: Acceptable diversification - Monitor geopolitical developments")
    
    print()

print("="*120)

In [13]:
print("\n" + "="*100)
print("PORTFOLIO DRIFT DETECTION SYSTEM - RM DASHBOARD SUMMARY")
print("Date: April 2026 | Crisis: IT Sector Decline + Geopolitical Risk")
print("="*100)

dashboard_data = []

for client_id, agent_info in drift_agents.items():
    portfolio = agent_info['portfolio']
    
    dashboard_data.append({
        'Client': portfolio['name'],
        'RM': portfolio['rm_name'],
        'Risk Profile': portfolio['risk_profile'],
        'Portfolio Value (₹)': f"{portfolio['portfolio_value']:,}",
        'IT Exposure %': f"{portfolio['target_allocation']['IT_stocks']*100:.0f}%",
        'Energy Exposure %': f"{portfolio['target_allocation']['energy_stocks']*100:.0f}%",
        'Agent ID': agent_info['agent'].id[:8] + '...',
        'Status': '🚨 ALERT' if portfolio['target_allocation']['IT_stocks'] > 0.30 or portfolio['target_allocation']['energy_stocks'] > 0.20 else '⚠️ MONITOR'
    })

df_dashboard = pd.DataFrame(dashboard_data)
print("\n" + df_dashboard.to_string(index=False))

print("\n" + "="*100)
print("ALERTS REQUIRING IMMEDIATE ATTENTION:")
print("="*100)

alert_count = 0
for client_id, agent_info in drift_agents.items():
    portfolio = agent_info['portfolio']
    it_exposure = portfolio['target_allocation']['IT_stocks']
    energy_exposure = portfolio['target_allocation']['energy_stocks']
    
    if it_exposure > 0.30:
        alert_count += 1
        print(f"\n🔴 CRITICAL - {portfolio['name']} ({client_id})")
        print(f"   IT Exposure: {it_exposure*100:.0f}% (CRITICAL - 8-12% decline in recent days)")
        print(f"   Estimated Impact: ₹{portfolio['portfolio_value'] * it_exposure * 0.10:,.0f}")
        print(f"   Action: IMMEDIATE REBALANCING NEEDED")
        print(f"   RM: {portfolio['rm_name']} - Prepare client call ASAP")
    
    if energy_exposure > 0.20:
        alert_count += 1
        print(f"\n🟠 HIGH - {portfolio['name']} ({client_id})")
        print(f"   Energy Exposure: {energy_exposure*100:.0f}% (At Risk - Iran-USA tensions)")
        print(f"   Estimated Impact: ₹{portfolio['portfolio_value'] * energy_exposure * 0.07:,.0f}")
        print(f"   Action: Monitor closely, prepare rebalancing plan")
        print(f"   RM: {portfolio['rm_name']}")

print(f"\n\n{'='*100}")
print(f"DRIFT AGENTS DEPLOYED: {len(drift_agents)}")
print(f"TOTAL PORTFOLIO VALUE UNDER MANAGEMENT: ₹{sum(p['portfolio_value'] for p in client_portfolios.values()):,}")
print(f"ALERTS GENERATED: {alert_count}")
print(f"{'='*100}")


PORTFOLIO DRIFT DETECTION SYSTEM - RM DASHBOARD SUMMARY
Date: April 2026 | Crisis: IT Sector Decline + Geopolitical Risk

                          Client                     RM  Risk Profile Portfolio Value (₹) IT Exposure % Energy Exposure %    Agent ID     Status
        Tech-Heavy Growth Client Relationship Manager A          High           5,000,000           45%               15% drift-de...    🚨 ALERT
          Balanced Growth Client Relationship Manager B      Moderate           7,500,000           25%               20% drift-de... ⚠️ MONITOR
      Conservative Income Client Relationship Manager C           Low           3,000,000           10%               15% drift-de... ⚠️ MONITOR
   Emerging Market Growth Client Relationship Manager D          High           4,000,000           30%               25% drift-de...    🚨 ALERT
Sector-Focused Specialist Client Relationship Manager E Moderate-High           6,000,000           40%               20% drift-de...    🚨 ALERT

ALERTS

### 12. Drift Agent Configuration Summary

In [14]:
print("\n" + "="*100)
print("PORTFOLIO DRIFT DETECTION AGENT SYSTEM - CONFIGURATION")
print("="*100)

config_summary = {
    'System Type': 'Portfolio Drift Detection & Alert Generation',
    'Purpose': 'Pre-Review stage automation for Private Banking',
    'Agents Deployed': len(drift_agents),
    'Total Clients': len(client_portfolios),
    'Total AUM': f"₹{sum(p['portfolio_value'] for p in client_portfolios.values()):,}",
    'Monitoring Parameters': 'Sector performance, geopolitical events, market indices',
    'Alert Thresholds': 'CRITICAL (>10% impact), HIGH (5-10%), MEDIUM (2-5%), LOW (<2%)',
    'Data Sources': 'Yahoo Finance (stocks), MFAPI (mutual funds), Market indices',
    'Framework': 'Microsoft Foundry with PromptAgentDefinition',
    'Update Frequency': 'Real-time monitoring with daily deep analysis'
}

for key, value in config_summary.items():
    print(f"{key:.<40} {value}")

print("\n" + "="*100)
print("NEXT STEPS - FULL APPLICATION ARCHITECTURE:")
print("="*100)
print("""
1. AGENT LAYER (Complete) ✅
   - Portfolio Drift Detection Agents created
   - Multi-turn conversation capability
   - Alert generation logic implemented

2. DYNAMICS 365 INTEGRATION (Next Phase)
   - Connect to CRM for life events
   - Store drift history and alerts
   - Track rebalancing actions

3. AZURE SERVICES STACK (Next Phase)
   - Azure Service Bus: Real-time alert notifications
   - Azure Cosmos DB: Portfolio data storage
   - Azure SQL Database: Historical analytics
   - Azure Functions: Scheduled drift calculations
   - Azure Logic Apps: RM notification workflows

4. DASHBOARD (Next Phase)
   - Power BI: Visual drift analysis
   - Power Apps: Mobile alerts for RMs
   - Real-time notifications

5. REPORTING (Next Phase)
   - Pre-review reports for RMs
   - Client communication templates
   - Recommended action summaries
""")
print("="*100)


PORTFOLIO DRIFT DETECTION AGENT SYSTEM - CONFIGURATION
System Type............................. Portfolio Drift Detection & Alert Generation
Purpose................................. Pre-Review stage automation for Private Banking
Agents Deployed......................... 5
Total Clients........................... 5
Total AUM............................... ₹25,500,000
Monitoring Parameters................... Sector performance, geopolitical events, market indices
Alert Thresholds........................ CRITICAL (>10% impact), HIGH (5-10%), MEDIUM (2-5%), LOW (<2%)
Data Sources............................ Yahoo Finance (stocks), MFAPI (mutual funds), Market indices
Framework............................... Microsoft Foundry with PromptAgentDefinition
Update Frequency........................ Real-time monitoring with daily deep analysis

NEXT STEPS - FULL APPLICATION ARCHITECTURE:

1. AGENT LAYER (Complete) ✅
   - Portfolio Drift Detection Agents created
   - Multi-turn conversation capab

### System Design Reference

Refer to the agent architecture diagram below for the complete workflow of the specialized portfolio drift detection agents, optimization agents, risk assessment agents, and recommendation agents:

![Agent System Design](../../assets/diagram-export-4-2-2026-10_53_26-PM.png)

### 12A. Advanced Finance Stack Configuration & Features

In [ ]:
print("\n" + "="*120)
print("ADVANCED PORTFOLIO DRIFT DETECTION SYSTEM - COMPLETE CONFIGURATION")
print("="*120)

system_config = {
    '🎯 PRIMARY OBJECTIVE': 'Private Banking Pre-Review Stage Automation with AI Agent Orchestration',
    
    '📊 DATA SOURCES': {
        'Market Data': 'Yahoo Finance (6-month historical)',
        'Portfolio Data': 'Client holdings in IT, Banking, Energy, Mutual Funds',
        'Crisis Baseline': 'Feb 2026 (actual date reference)',
        'Crisis Scenario': 'Apr 2026 (IT -8-12%, Energy -5-7%, Banking -3-5%)'
    },
    
    '💰 FINANCE PACKAGES INTEGRATED': {
        'PyPortfolioOpt': 'Efficient Frontier optimization, Sharpe maximization, Discrete allocation',
        'VectorBT': 'Vectorized backtesting, scenario analysis, performance metrics',
        'SciPy': 'Statistical analysis, correlation matrices, risk metrics (VaR, CVaR)',
        'Skfolio': 'Advanced portfolio optimization (framework ready for advanced models)',
        'YFinance': 'Real-time market data, historical OHLCV',
        'Scikit-Learn': 'PCA for correlation analysis, normalization'
    },
    
    '🤖 AGENT ARCHITECTURE': {
        'Total Agents': f'{len(drift_agents)} Drift Detection Agents',
        'Specialized Agents': '5 (Data, Optimizer, Risk, Detector, Recommendation)',
        'Pattern': 'Multi-Agent Workflow with Sequential & Parallel execution',
        'Conversation Type': 'Multi-turn with context preservation'
    },
    
    '📈 ANALYSIS CAPABILITIES': {
        'Portfolio Optimization': '✅ Efficient Frontier, Modern Portfolio Theory',
        'Risk Metrics': '✅ VaR, CVaR, Sharpe, Sortino, Max Drawdown, Calmar Ratio',
        'Stress Testing': '✅ 6 crisis scenarios, correlation spike analysis',
        'Backtesting': '✅ Performance under crisis, scenario impact projection',
        'Drift Detection': '✅ Target vs Optimal vs Current allocation comparison'
    },
    
    '👥 CLIENT COVERAGE': {
        'Total Clients': len(client_portfolios),
        'Total AUM': f"₹{sum(p['portfolio_value'] for p in client_portfolios.values())/10000000:.1f}Cr",
        'Risk Profiles': 'High, Moderate-High, Moderate, Low',
        'Data Available': f'{len(client_optimization_data)} clients with complete analysis'
    },
    
    '⚠️  ALERT TRIGGERS': {
        'CRITICAL': 'Drift >10% from target OR Sharpe <50% of baseline OR Portfolio impact >₹50L',
        'HIGH': 'Drift 5-10% OR Sharpe 50-75% OR Portfolio impact ₹25-50L',
        'MEDIUM': 'Drift 2-5% OR Sharpe 75-90% OR Portfolio impact ₹10-25L',
        'LOW': 'Drift <2% OR Sharpe >90% OR Portfolio impact <₹10L'
    },
    
    '🔄 WORKFLOW STAGES': {
        'Stage 1': 'Data Collection & Statistical Analysis',
        'Stage 2': 'Modern Portfolio Theory Optimization',
        'Stage 3': 'Risk Assessment & Stress Testing',
        'Stage 4': 'Drift Detection & Deviation Analysis',
        'Stage 5': 'Recommendation Generation',
        'Stage 6': 'RM Dashboard Aggregation'
    },
    
    '📊 KEY METRICS COMPUTED': [
        'Expected Annual Return, Volatility, Sharpe Ratio',
        'Value at Risk (95%), Conditional Value at Risk',
        'Maximum Drawdown, Sortino Ratio, Calmar Ratio',
        'Correlation Matrix, Covariance Matrix',
        'Portfolio Skewness, Kurtosis, Win Rate',
        'Scenario-wise Impact, Resilience Score'
    ],
    
    '🎯 RM DASHBOARD OUTPUTS': [
        'Feb 2026 Baseline: Expected return, volatility, Sharpe ratio',
        'Optimal Allocation: Efficient Frontier recommendations',
        'Crisis Impact: Worst-case scenario analysis',
        'Drift Status: Target vs Optimal comparison',
        'Action Items: Specific rebalancing recommendations',
        'Expected Outcome: Projected Sharpe improvement'
    ]
}

print("\n📋 SYSTEM CONFIGURATION SUMMARY:\n")

for section, content in system_config.items():
    print(f"{section}")
    print("-" * 100)
    
    if isinstance(content, dict):
        for key, value in content.items():
            if isinstance(value, str):
                print(f"  {key:.<35} {value}")
            else:
                print(f"  {key}:")
                for sub_key, sub_value in value.items():
                    print(f"    • {sub_key}: {sub_value}")
    elif isinstance(content, list):
        for item in content:
            print(f"  • {item}")
    else:
        print(f"  {content}")
    
    print()

print("="*120)
print("NEXT PHASE: FULL AZURE ARCHITECTURE INTEGRATION")
print("="*120)
print("""
Ready to Deploy:
✅ Agent Layer: 5 Drift Detection Agents + 5 Specialized Agents
✅ Data Layer: Feb 2026 Historical Analysis + Apr 2026 Crisis Simulation
✅ Analysis Layer: Portfolio Optimization + Risk Assessment + Stress Testing
✅ Intelligence Layer: Multi-turn conversations with context preservation

Next Steps for Complete Application:
1. DYNAMICS 365 INTEGRATION: CRM for client life events, rebalancing history
2. AZURE SERVICES STACK:
   - Service Bus: Real-time alerts to RMs
   - Cosmos DB: Historical portfolio data, drift events
   - SQL Database: Analytics, performance metrics
   - Functions: Daily drift calculation triggers
   - Logic Apps: RM workflow automation
3. POWER BI: Visual dashboard for drift analytics
4. POWER APPS: Mobile notifications for RM alerts
5. REPORTING: Pre-review reports, client communication templates
""")
print("="*120)